In [0]:
#PySpark JSON, Struct & Array Operations

In [0]:
 from pyspark.sql.functions import *


json_data = [
    {
        "emp_id": 101,
        "name": "Arun",
        "department": "IT",
        "address": {
            "city": "Chennai",
            "state": "Tamil Nadu",
            "pincode": 600001
        },
        "skills": ["Python", "PySpark", "SQL"],
        "projects": [
            {
                "project_id": "P101",
                "project_name": "Banking",
                "duration": 12
            },
            {
                "project_id": "P102",
                "project_name": "Retail",
                "duration": 8
            }
        ]
    },
    {
        "emp_id": 102,
        "name": "Priya",
        "department": "HR",
        "address": {
            "city": "Bangalore",
            "state": "Karnataka",
            "pincode": 560001
        },
        "skills": ["Excel", "SQL", "Power BI"],
        "projects": [
            {
                "project_id": "P103",
                "project_name": "HR Analytics",
                "duration": 10
            }
        ]
    },
    {
        "emp_id": 103,
        "name": "Rahul",
        "department": "IT",
        "address": {
            "city": "Chennai",
            "state": "Tamil Nadu",
            "pincode": 600002
        },
        "skills": ["Python", "Databricks", "Spark"],
        "projects": [
            {
                "project_id": "P104",
                "project_name": "Data Platform",
                "duration": 18
            },
            {
                "project_id": "P105",
                "project_name": "Migration",
                "duration": 6
            }
        ]
    },
    {
        "emp_id": 104,
        "name": "Divya",
        "department": "Finance",
        "address": {
            "city": "Mumbai",
            "state": "Maharashtra",
            "pincode": 400001
        },
        "skills": ["SQL", "Power BI", "Excel"],
        "projects": [
            {
                "project_id": "P106",
                "project_name": "Financial Reporting",
                "duration": 14
            }
        ]
    }
]

df = spark.createDataFrame(json_data)


df.show(truncate=False)

+----------------------------------------------------------+----------+------+-----+-------------------------------------------------------------------------------------------------------------------------------------+---------------------------+
|address                                                   |department|emp_id|name |projects                                                                                                                             |skills                     |
+----------------------------------------------------------+----------+------+-----+-------------------------------------------------------------------------------------------------------------------------------------+---------------------------+
|{city -> Chennai, state -> Tamil Nadu, pincode -> 600001} |IT        |101   |Arun |[{project_id -> P101, project_name -> Banking, duration -> 12}, {project_id -> P102, project_name -> Retail, duration -> 8}]         |[Python, PySpark, SQL]     |
|{city -> Ba

In [0]:
#1. Explore the JSON Schema
df.printSchema()

root
 |-- address: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- department: string (nullable = true)
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- projects: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: string (valueContainsNull = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
#2. Select Nested Address Fields
df.select("emp_id","name","address.city","address.state","address.pincode").show()

+------+-----+---------+-----------+-------+
|emp_id| name|     city|      state|pincode|
+------+-----+---------+-----------+-------+
|   101| Arun|  Chennai| Tamil Nadu| 600001|
|   102|Priya|Bangalore|  Karnataka| 560001|
|   103|Rahul|  Chennai| Tamil Nadu| 600002|
|   104|Divya|   Mumbai|Maharashtra| 400001|
+------+-----+---------+-----------+-------+



In [0]:
#rename
from pyspark.sql.functions import col

df.select(
    "emp_id",
    "name",
    col("address.city").alias("city"),
    col("address.state").alias("state"),
    col("address.pincode").alias("pincode")
).show()

+------+-----+---------+-----------+-------+
|emp_id| name|     city|      state|pincode|
+------+-----+---------+-----------+-------+
|   101| Arun|  Chennai| Tamil Nadu| 600001|
|   102|Priya|Bangalore|  Karnataka| 560001|
|   103|Rahul|  Chennai| Tamil Nadu| 600002|
|   104|Divya|   Mumbai|Maharashtra| 400001|
+------+-----+---------+-----------+-------+



In [0]:
#3. Select Employee Skills
from pyspark.sql.functions import *
df.select("name","skills").show(truncate = False)
df.select("name",col("skills")[0].alias("skill_one")).show(truncate = False)

+-----+---------------------------+
|name |skills                     |
+-----+---------------------------+
|Arun |[Python, PySpark, SQL]     |
|Priya|[Excel, SQL, Power BI]     |
|Rahul|[Python, Databricks, Spark]|
|Divya|[SQL, Power BI, Excel]     |
+-----+---------------------------+

+-----+---------+
|name |skill_one|
+-----+---------+
|Arun |Python   |
|Priya|Excel    |
|Rahul|Python   |
|Divya|SQL      |
+-----+---------+



In [0]:
#4.Find Number of Skills
from pyspark.sql.functions import *
df.withColumn("skills_count",size(col("skills"))).select("name","skills","skills_count").show()

+-----+--------------------+------------+
| name|              skills|skills_count|
+-----+--------------------+------------+
| Arun|[Python, PySpark,...|           3|
|Priya|[Excel, SQL, Powe...|           3|
|Rahul|[Python, Databric...|           3|
|Divya|[SQL, Power BI, E...|           3|
+-----+--------------------+------------+



In [0]:
#5. Filter Based on Nested Field
df.filter(df.address.city == "Chennai").select("name","address.city").show()

+-----+-------+
| name|   city|
+-----+-------+
| Arun|Chennai|
|Rahul|Chennai|
+-----+-------+



In [0]:
#6. Filter Employees Based on Array
from pyspark.sql.functions import *
df.filter(array_contains("skills", "Python")).select("name", "skills").show(truncate=False)

+-----+---------------------------+
|name |skills                     |
+-----+---------------------------+
|Arun |[Python, PySpark, SQL]     |
|Rahul|[Python, Databricks, Spark]|
+-----+---------------------------+



In [0]:
#7.Extract Project Information first project at index
df.select("name","projects").show(truncate =False)

df.select("name",df.projects[0].project_id.alias("project_id"),df.projects[0].project_name.alias("project_name"),df.projects[0].duration.alias("duration")).show()

+-----+-------------------------------------------------------------------------------------------------------------------------------------+
|name |projects                                                                                                                             |
+-----+-------------------------------------------------------------------------------------------------------------------------------------+
|Arun |[{project_id -> P101, project_name -> Banking, duration -> 12}, {project_id -> P102, project_name -> Retail, duration -> 8}]         |
|Priya|[{project_id -> P103, project_name -> HR Analytics, duration -> 10}]                                                                 |
|Rahul|[{project_id -> P104, project_name -> Data Platform, duration -> 18}, {project_id -> P105, project_name -> Migration, duration -> 6}]|
|Divya|[{project_id -> P106, project_name -> Financial Reporting, duration -> 14}]                                                          |
+-----

In [0]:
#8.Explode the Projects Array
project_df = df.withColumn("project", explode("projects")) #create new column convert project into row
#project_df.show(truncate=False) 
project_df.select("name","project.project_id","project.project_name","project.duration").show(truncate=False) #it will extract projects 

   

+-----+----------+-------------------+--------+
|name |project_id|project_name       |duration|
+-----+----------+-------------------+--------+
|Arun |P101      |Banking            |12      |
|Arun |P102      |Retail             |8       |
|Priya|P103      |HR Analytics       |10      |
|Rahul|P104      |Data Platform      |18      |
|Rahul|P105      |Migration          |6       |
|Divya|P106      |Financial Reporting|14      |
+-----+----------+-------------------+--------+



In [0]:
#9. Find Long-Duration Projects
project_df1 = df.withColumn("Long_DP",explode("projects"))
project_df1.filter(project_df1.Long_DP.duration > 10).select("name","Long_DP.project_name","Long_DP.duration").show()

+-----+-------------------+--------+
| name|       project_name|duration|
+-----+-------------------+--------+
| Arun|            Banking|      12|
|Rahul|      Data Platform|      18|
|Divya|Financial Reporting|      14|
+-----+-------------------+--------+



In [0]:
#10.Advanced Nested JSON Challenge
f_df = (df.withColumn("city",col("address.city")) #it will extract city from address
          .withColumn("state",col("address.state")) #it will extract state from address
          .withColumn("skill_count",size(col("skills"))) #it will give skills count
          .withColumn("project",explode("projects")) #it will array into row
          .withColumn("project_name",col("project.project_name")) #it will extract project_name from project
          .withColumn("project_duration",col("project.duration")) #it will extract project_duration from project
          .filter("project_duration > 8")
          .select(
               "emp_id",
               "name",
               "city",
               "state",
               "skill_count",
               "project_name",
               "project_duration"
            )
            .orderBy(col("project_duration").desc())
)

f_df.show(truncate=False)
            

+------+-----+---------+-----------+-----------+-------------------+----------------+
|emp_id|name |city     |state      |skill_count|project_name       |project_duration|
+------+-----+---------+-----------+-----------+-------------------+----------------+
|103   |Rahul|Chennai  |Tamil Nadu |3          |Data Platform      |18              |
|104   |Divya|Mumbai   |Maharashtra|3          |Financial Reporting|14              |
|101   |Arun |Chennai  |Tamil Nadu |3          |Banking            |12              |
|102   |Priya|Bangalore|Karnataka  |3          |HR Analytics       |10              |
+------+-----+---------+-----------+-----------+-------------------+----------------+

